In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

/home/adellorto/Projects/MiniGPT/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
from tokenizer import CharachterLevelTokenizer, TiktokenTokenizer

In [3]:
# read it in to inspect it
with open('data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(text[:200])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [4]:
char_tokenizer = CharachterLevelTokenizer(text)
tiktoken_tokenizer = TiktokenTokenizer(text)

print(f"CharTokenizer vocab size:       {len(char_tokenizer.vocab)}")
print(f"Tiktoken compact vocab size:    {len(tiktoken_tokenizer.vocab)}")
print(f"Tiktoken full vocab size:       {tiktoken_tokenizer.encoding.n_vocab}")

CharTokenizer vocab size:       65
Tiktoken compact vocab size:    12111
Tiktoken full vocab size:       100277


In [5]:
base_params = {
    'batch_size' : 32,
    'block_size' : 8,
    'max_iters' : 3000,
    'eval_interval' : 300,
    'learning_rate' : 1e-2,
    'eval_iters' : 200  
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(42)

In [6]:
def get_batch(data, params, device):
    ix = torch.randint(len(data) - params['block_size'], (params['batch_size'],))
    x = torch.stack([data[i:i+params['block_size']] for i in ix])
    y = torch.stack([data[i+1:i+params['block_size']+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [7]:
@torch.no_grad()
def estimate_loss(model, data, params, device):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(params['eval_iters'])
        for k in range(params['eval_iters']):
            X, Y = get_batch(data[split], params, device)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [8]:
# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


In [9]:
from tqdm import tqdm

def run_experiment(tokenizer, name):
    # Encode full text first
    encoded = torch.tensor(tokenizer.encode(text), dtype=torch.long)

    vocab_size = len(tokenizer.vocab)
    params = {**base_params, 'vocab_size': vocab_size}

    # Train/val split
    n = int(0.9 * len(encoded))
    data = {'train': encoded[:n], 'val': encoded[n:]}

    # Model + optimizer
    torch.manual_seed(42)
    model = BigramLanguageModel(vocab_size).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params['learning_rate'])

    print(f"\n=== {name} | vocab_size={vocab_size} | tokens={len(encoded)} ===")
    history = []
    pbar = tqdm(range(params['max_iters'] + 1), desc=name)
    for iter in pbar:
        if iter % params['eval_interval'] == 0:
            losses = estimate_loss(model, data, params, device)
            pbar.set_postfix(train=f"{losses['train']:.4f}", val=f"{losses['val']:.4f}")
            tqdm.write(f"  step {iter:4d}: train {losses['train']:.4f}  val {losses['val']:.4f}")
            history.append({'step': iter, 'train': losses['train'].item(), 'val': losses['val'].item()})
        if iter < params['max_iters']:
            xb, yb = get_batch(data['train'], params, device)
            _, loss = model(xb, yb)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

    return model, history

In [10]:
char_model, char_history = run_experiment(char_tokenizer, "CharacterLevel")


=== CharacterLevel | vocab_size=65 | tokens=1115394 ===


CharacterLevel:   1%|          | 30/3001 [00:01<01:19, 37.48it/s, train=4.7627, val=4.7633]

  step    0: train 4.7627  val 4.7633


CharacterLevel:  12%|█▏        | 358/3001 [00:02<00:18, 141.53it/s, train=2.8415, val=2.8635]

  step  300: train 2.8415  val 2.8635


...

  step 2400: train 2.4637  val 2.4956


CharacterLevel:  91%|█████████▏| 2742/3001 [00:12<00:01, 177.38it/s, train=2.4644, val=2.4894]

  step 2700: train 2.4644  val 2.4894


CharacterLevel: 100%|██████████| 3001/3001 [00:13<00:00, 216.50it/s, train=2.4624, val=2.4934]

  step 3000: train 2.4624  val 2.4934


In [11]:
tiktoken_model, tiktoken_history = run_experiment(tiktoken_tokenizer, "Tiktoken (cl100k)")


=== Tiktoken (cl100k) | vocab_size=12111 | tokens=301829 ===


Tiktoken (cl100k):   0%|          | 0/3001 [00:04<?, ?it/s, train=9.9066, val=9.8802]

  step    0: train 9.9066  val 9.8802


Tiktoken (cl100k):  10%|▉         | 300/3001 [05:16<40:23,  1.11it/s, train=8.6783, val=8.8184]  

  step  300: train 8.6783  val 8.8184


Tiktoken (cl100k):  12%|█▏        | 375/3001 [06:26<45:08,  1.03s/it, train=8.6783, val=8.8184]  


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

for ax, history, name in zip(axes, [char_history, tiktoken_history], ["CharacterLevel", "Tiktoken (cl100k)"]):
    steps = [h['step'] for h in history]
    ax.plot(steps, [h['train'] for h in history], label='train')
    ax.plot(steps, [h['val'] for h in history], label='val', linestyle='--')
    ax.set_title(name)
    ax.set_xlabel('step')
    ax.set_ylabel('loss')
    ax.legend()

plt.suptitle('Bigram Model: Loss Comparison by Tokenizer')
plt.tight_layout()
plt.show()

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)

print("=== CharacterLevel generation ===")
tokens = char_model.generate(context, max_new_tokens=500)[0].tolist()
print(char_tokenizer.decode(tokens))

print("\n=== Tiktoken generation ===")
tokens = tiktoken_model.generate(context, max_new_tokens=500)[0].tolist()
print(tiktoken_tokenizer.decode(tokens))